# Rebalancear a carteira vale a pena?

**Python Finance Lab · Aula 4** · *Finanças na prática, notebook por notebook.*

---

## Parte 1 · A pergunta

Nas aulas 1 e 3, a regra foi simples: dividir o dinheiro em partes iguais entre as 10 ações e não mexer mais. Só que o mercado mexe. As ações que sobem passam a ocupar uma fatia cada vez maior da carteira, e as que caem, uma fatia cada vez menor. Depois de alguns anos, a carteira que nasceu com 10% em cada ação já é outra carteira, mais concentrada nas vencedoras do passado.

**Rebalancear** é trazer os pesos de volta ao alvo. Existem duas formas. A primeira é vender um pouco do que subiu e comprar do que caiu. A segunda não vende nada: usa o dinheiro novo de cada mês para reforçar o que ficou para trás.

Nesta aula, vamos comparar quatro estratégias com exatamente o mesmo dinheiro: R\\$ 2.000 no primeiro dia, divididos igualmente, e R\\$ 200 no fim de cada mês, de outubro de 2021 a setembro de 2026. São R\\$ 14.000 aportados no total.

- **Nunca:** compra e segura; os aportes são divididos igualmente, como na aula 3.
- **Mensal:** todo fim de mês, vende o que passou de 10% e compra o que ficou abaixo.
- **Anual:** o mesmo, mas só no fim de dezembro.
- **Aporte:** não vende nada; o aporte do mês vai inteiro para a ação mais abaixo do peso.

Do lado do Python, o código aprende a **tomar decisões** com `if`, `elif` e `else` ("se é mês de rebalancear, então..."), e ganha uma estrutura nova para guardar resultados pelo nome: o **dicionário**.

*Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*

## Parte 2 · Conceitos

### Bloco 2 · O vocabulário da aula

**Peso.** É a fatia de cada ação no patrimônio da carteira. Se a carteira vale R\\$ 1.000 e as ações do Itaú valem R\\$ 130, o peso do Itaú é 13%. Os pesos de todas as ações sempre somam 100%.

**Deriva dos pesos.** É o afastamento natural dos pesos em relação ao alvo, causado pelas próprias variações de preço. Ninguém precisa fazer nada para a carteira derivar: basta o tempo passar.

**Rebalanceamento por venda.** Vender um pouco das ações acima do alvo e usar o dinheiro para comprar as que estão abaixo, até todas voltarem a 10%. Na prática, é "vender o que subiu e comprar o que caiu", o contrário do que o instinto costuma pedir.

**Rebalanceamento por aporte.** Em vez de vender, direcionar o dinheiro novo para as ações que ficaram para trás. É mais lento, porque depende do tamanho do aporte em relação à carteira, mas não gera vendas.

**Giro.** É quanto dinheiro trocou de mãos num rebalanceamento: o valor vendido, que é igual ao valor comprado. Quanto maior o giro, mais a carteira foi mexida.

**Custo de negociar.** Toda compra e toda venda têm algum custo: taxas da bolsa, eventual corretagem e a diferença entre o preço de compra e o de venda no momento da ordem. Vamos usar como exemplo **0,1% sobre o valor de cada compra e de cada venda**.

**Imposto.** Vender uma ação com lucro pode gerar imposto de renda. As regras têm limites e exceções, e mudam com o tempo, por isso esta aula não calcula imposto. Mas vale guardar um fato: o rebalanceamento por aporte não vende nada e, portanto, não gera esse imposto.

**Cota da carteira.** Para comparar estratégias que recebem aportes, não dá para olhar só o patrimônio, porque ele cresce também com os depósitos. A solução é a mesma dos fundos de investimento, que conhecemos na aula 3: acompanhar o valor de uma **cota**, que muda só com o rendimento. O retorno da cota no mês é o crescimento do patrimônio descontando o aporte daquele mês.

## Parte 3 · Fórmulas

### Bloco 3 · As contas da aula

**Peso** da ação $i$ no dia $t$, com $V_{i,t}$ sendo o valor investido nela:

$$w_{i,t} = \frac{V_{i,t}}{\sum_j V_{j,t}}$$

**Depois do rebalanceamento**, cada uma das $N$ ações fica com a mesma parte do total:

$$V^{\text{depois}}_{i,t} = \frac{1}{N}\sum_j V_{j,t}$$

**Giro**, o valor vendido (igual ao valor comprado):

$$G_t = \frac{1}{2}\sum_i \left|V^{\text{depois}}_{i,t} - V^{\text{antes}}_{i,t}\right|$$

**Custo**, com taxa $c$ sobre cada venda e cada compra: $c \times 2\,G_t$.

**Retorno da cota** no mês $m$, com patrimônio $W_m$ e aporte $A$ feito no fim do mês:

$$r_m = \frac{W_m - A}{W_{m-1}} - 1 \qquad\qquad C_m = C_{m-1}\,(1 + r_m), \quad C_0 = 1$$

*Exemplo à mão:* uma carteira com duas ações, R\\$ 100 em cada. A ação A sobe para R\\$ 150 e a ação B cai para R\\$ 90. O total é R\\$ 240, e os pesos passam a ser 62,5% e 37,5%.

- **Rebalancear vendendo:** o alvo é R\\$ 120 em cada uma. Vendemos R\\$ 30 de A e compramos R\\$ 30 de B. O giro é R\\$ 30, e o custo é 0,1% × R\\$ 60 = R\\$ 0,06.
- **Rebalancear com um aporte de R\\$ 20:** o aporte vai inteiro para B, a mais abaixo do peso. A fica com R\\$ 150 e B com R\\$ 110; os pesos passam a 57,7% e 42,3%. Mais perto do alvo, sem vender nada.

*Exemplo da cota:* se o patrimônio era R\\$ 240, o aporte do mês foi de R\\$ 20 e o patrimônio terminou o mês em R\\$ 272, o retorno da cota foi $(272 - 20)/240 - 1 = 5\%$. Os outros R\\$ 20 de crescimento vieram do depósito, e não do rendimento.

## Parte 4 · Implementação

### Bloco 4 · Importar as bibliotecas

As mesmas bibliotecas das aulas anteriores.

> **O que esperar:** nada. Se aparecer `ModuleNotFoundError`, instale com `%pip install yfinance pandas matplotlib requests` e reinicie o kernel (menu Kernel → Restart Kernel).

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests

### Bloco 5 · As premissas

As mesmas ações e datas de sempre, o capital inicial da aula 1, o aporte mensal da aula 3 e uma premissa nova: a taxa de custo por negociação. O Ibovespa não entra nesta aula, porque vamos comparar a carteira com ela mesma, em quatro versões.

> **O que esperar:** nada.

In [ ]:
ACOES = [
    "ITUB4.SA", "VALE3.SA", "PETR4.SA", "WEGE3.SA", "ABEV3.SA",
    "EGIE3.SA", "SUZB3.SA", "RADL3.SA", "VIVT3.SA", "TOTS3.SA",
]
DATA_INICIO = "2021-10-01"
DATA_FIM = "2026-09-30"
CAPITAL = 2000          # R$ 2.000 no primeiro dia
APORTE_MENSAL = 200     # R$ 200 no fim de cada mês
TAXA = 0.001            # custo de 0,1% sobre o valor de cada compra e de cada venda

### Bloco 6 · Os dados

O bloco de dados das aulas anteriores, sem o Ibovespa: os preços ajustados das 10 ações, do Yahoo Finance, e o CDI diário, do Banco Central, que será a régua do índice de Sharpe.

> **O que esperar:** nada. Os dados ficam em `precos` e `cdi_diario`.

In [ ]:
# Ações (Yahoo Finance), como na aula 1
dados = yf.download(ACOES, start=DATA_INICIO, auto_adjust=True, progress=False)
precos = dados["Close"].loc[DATA_INICIO:DATA_FIM].ffill()
precos.columns = precos.columns.str.replace(".SA", "")

# CDI (Banco Central), como na aula 1
inicio_bc = pd.to_datetime(DATA_INICIO).strftime("%d/%m/%Y")
fim_bc = pd.to_datetime(DATA_FIM).strftime("%d/%m/%Y")
url = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
    f"?formato=json&dataInicial={inicio_bc}&dataFinal={fim_bc}"
)
resposta = requests.get(url, timeout=60)
resposta.raise_for_status()
cdi_bruto = pd.DataFrame(resposta.json())
cdi_bruto["data"] = pd.to_datetime(cdi_bruto["data"], format="%d/%m/%Y")
cdi_bruto["valor"] = cdi_bruto["valor"].astype(float)
cdi_diario = cdi_bruto.set_index("data")["valor"]

### Bloco 7 · A deriva dos pesos

Antes de rebalancear, vale ver o problema. Voltamos à carteira da aula 1, os R\\$ 2.000 sem aportes e sem rebalanceamento, e calculamos o peso de cada ação em cada dia.

- `valores` é quanto valem os R\\$ 200 iniciais de cada ação, dia a dia: a conta do bloco 13 da aula 1 multiplicada por R\\$ 200.
- `valores.sum(axis=1)` é o patrimônio de cada dia, a soma das 10 colunas.
- `.div(..., axis=0)` divide cada linha pelo patrimônio daquele dia. É o mesmo alinhamento por datas do `.sub(..., axis=0)` da aula 3.

Desenhar 10 pesos num gráfico só daria um emaranhado de linhas. Por isso, o gráfico mostra a **maior** e a **menor** fatia de cada dia, com `max(axis=1)` e `min(axis=1)`. No primeiro dia, as duas linhas saem juntas de 10%; a distância entre elas é a deriva.

> **O que esperar:** duas linhas que saem juntas de 10% e se afastam com o tempo, uma para cima e outra para baixo. A linha cinza marca o alvo de 10%. O `ax=ax` faz a segunda linha ser desenhada no mesmo gráfico da primeira.

In [ ]:
valores = precos / precos.iloc[0] * (CAPITAL / len(ACOES))
pesos = valores.div(valores.sum(axis=1), axis=0)

ax = (pesos.max(axis=1) * 100).plot(
    figsize=(10, 4), color="#2a78d6", linewidth=2, label="Maior peso"
)
(pesos.min(axis=1) * 100).plot(ax=ax, color="#2a78d6", linestyle="--", linewidth=2, label="Menor peso")
ax.axhline(10, color="gray", linewidth=0.8)
ax.set_title("Carteira da aula 1, sem rebalancear: a maior e a menor fatia")
ax.set_ylabel("Peso (%)")
ax.set_xlabel("")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Depois de rodar.** **[Preencher: qual foi a maior e a menor fatia no fim do período.]**

### Bloco 8 · O primeiro `if`

Agora, o peso de cada ação no último dia, com uma decisão para cada uma: está acima, abaixo ou no alvo? É o trabalho do `if`.

- `if peso > 0.10:` testa uma **comparação**. Se ela for verdadeira, roda o bloco recuado logo abaixo, e as outras alternativas são ignoradas.
- `elif`, abreviação de *else if* ("senão, se"), testa uma segunda condição, só quando a primeira foi falsa.
- `else` ("senão") pega todos os casos que sobraram.

As comparações do Python são `>` (maior), `<` (menor), `>=` (maior ou igual), `<=` (menor ou igual), `==` (igual) e `!=` (diferente). Repare que a igualdade usa dois sinais de igual: um só, `=`, guarda um valor numa variável.

Na f-string, `:.1%` mostra o número como porcentagem, com uma casa decimal: 0.1234 aparece como 12.3%.

> **O que esperar:** 10 linhas, uma por ação, com o peso final e a situação de cada uma. O caso "exatamente no alvo" quase nunca acontece com preços reais, mas o `else` deixa a decisão completa.

In [ ]:
pesos_finais = pesos.iloc[-1]

for acao in pesos_finais.index:
    peso = pesos_finais[acao]
    if peso > 0.10:
        situacao = "acima do alvo"
    elif peso < 0.10:
        situacao = "abaixo do alvo"
    else:
        situacao = "exatamente no alvo"
    print(f"{acao}: {peso:.1%} ({situacao})")

**Depois de rodar.** **[Preencher: quantas ações terminaram acima de 10%.]**

### Bloco 9 · As datas de decisão

A simulação acontece em 61 datas: o primeiro dia, quando os R\\$ 2.000 são investidos, e os 60 fins de mês, quando entra cada aporte e, se for o caso, acontece o rebalanceamento.

- `precos.resample("ME").last()` é o preço de fim de mês da aula 3.
- `precos.iloc[[0]]`, com **colchetes duplos**, pega a primeira linha como uma tabela de uma linha só, e não como uma Series. Assim, o `pd.concat` consegue empilhá-la em cima da tabela mensal.

Fazemos o mesmo com a cota do CDI, que será a régua do Sharpe.

> **O que esperar:** `Datas de decisão: 61` e as três primeiras linhas: 01/10/2021, 31/10/2021 e 30/11/2021.

In [ ]:
precos_mes = precos.resample("ME").last()
precos_decisao = pd.concat([precos.iloc[[0]], precos_mes])

cota_cdi = (1 + cdi_diario / 100).cumprod()
cdi_decisao = pd.concat([cota_cdi.iloc[[0]], cota_cdi.resample("ME").last()])

print("Datas de decisão:", len(precos_decisao))
precos_decisao.head(3)

### Bloco 10 · O dicionário

Uma lista guarda valores em ordem, e cada um é encontrado pela posição: `ACOES[0]`. Um **dicionário** guarda pares de **chave** e **valor**, e cada valor é encontrado pelo nome da chave. Ele é escrito entre chaves, `{}`, com dois-pontos entre a chave e o valor.

- `estrategias["anual"]` busca o valor guardado na chave `"anual"`.
- `.items()` devolve os pares, um de cada vez. No `for`, os dois nomes separados por vírgula, `nome, descricao`, recebem a chave e o valor de cada par.

Vamos usar dicionários de duas formas nesta aula: para descrever as estratégias e, mais adiante, para guardar o resultado de cada uma pelo nome.

> **O que esperar:** primeiro, a descrição da estratégia anual; depois, uma linha em branco e as quatro estratégias com suas descrições.

In [ ]:
estrategias = {
    "nunca": "compra no primeiro dia e aporta em partes iguais, sem vender",
    "mensal": "todo fim de mês, vende o que passou de 10% e compra o que ficou abaixo",
    "anual": "o mesmo, mas só no fim de dezembro",
    "aporte": "não vende nada: o aporte vai inteiro para a ação mais abaixo do peso",
}

print(estrategias["anual"])
print()
for nome, descricao in estrategias.items():
    print(f"{nome}: {descricao}")

### Bloco 11 · A função que simula as estratégias

Esta é a função mais longa da série até aqui, mas ela é só o laço da aula 3 com duas decisões dentro. Ela recebe os preços nas datas de decisão, o capital inicial, o aporte, o nome da estratégia e a taxa de custo. Começa comprando o capital inicial em partes iguais e, depois, percorre os fins de mês. Em cada um:

1. **O aporte.** Se a estratégia for `"aporte"`, calcula os pesos, encontra a ação mais atrasada com `idxmin()` (da aula 2) e compra só ela. Nas outras estratégias, divide o aporte em partes iguais, como na aula 3. A linha `cotas[atrasada] = ...` altera um único item da Series: o número de cotas daquela ação.
2. **A decisão de rebalancear.** Com `if`, `elif` e `else`: na mensal, sempre; na anual, só se o mês da data for dezembro (`data.month == 12`); nas outras, nunca. A comparação `data.month == 12` vale verdadeiro ou falso, e esse resultado fica guardado em `rebalancear`.
3. **O rebalanceamento.** Se `rebalancear` for verdadeiro, calcula o valor-alvo de cada ação, o valor vendido (o giro do mês), desconta o custo e recompra tudo em partes iguais.
4. **O registro.** Guarda o patrimônio e a cota do mês. O `if` da cota protege contra uma divisão por zero quando ainda não havia dinheiro investido, o que acontece se o capital inicial for zero.

Duas novidades na primeira linha da função:

- `taxa=0` é um **parâmetro com valor padrão**: se ninguém informar a taxa, ela vale zero.
- `precos.index[1:]` são todas as datas a partir da segunda, porque a primeira já foi usada na compra inicial.

No fim, a função devolve um dicionário com quatro resultados: o patrimônio e a cota mês a mês, o giro total e as cotas finais de cada ação.

> **O que esperar:** nada. A função foi ensinada ao Python; ainda não rodou.

In [ ]:
def simular_carteira(precos, inicial, aporte, estrategia, taxa=0):
    """Simula a carteira nas datas de `precos` e devolve um dicionário com os resultados."""
    n = len(precos.columns)
    cotas = (inicial / n) / precos.iloc[0]      # primeiro dia: partes iguais
    patrimonio = [inicial]
    cota = [1.0]
    giro = 0

    for data in precos.index[1:]:
        preco = precos.loc[data]

        # 1. O aporte do mês
        if estrategia == "aporte":
            valores = cotas * preco
            pesos = valores / valores.sum()
            atrasada = pesos.idxmin()
            cotas[atrasada] = cotas[atrasada] + aporte / preco[atrasada]
        else:
            cotas = cotas + (aporte / n) / preco

        # 2. É hora de rebalancear vendendo?
        if estrategia == "mensal":
            rebalancear = True
        elif estrategia == "anual":
            rebalancear = data.month == 12
        else:
            rebalancear = False

        # 3. O rebalanceamento
        if rebalancear:
            valores = cotas * preco
            alvo = valores.sum() / n
            vendido = (valores - alvo).abs().sum() / 2
            giro = giro + vendido
            total = valores.sum() - taxa * 2 * vendido
            cotas = (total / n) / preco

        # 4. Patrimônio e cota no fim do mês
        valor = (cotas * preco).sum()
        if patrimonio[-1] > 0:
            cota.append(cota[-1] * (valor - aporte) / patrimonio[-1])
        else:
            cota.append(cota[-1])
        patrimonio.append(valor)

    return {
        "patrimonio": pd.Series(patrimonio, index=precos.index),
        "cota": pd.Series(cota, index=precos.index),
        "giro": giro,
        "cotas": cotas,
    }

### Bloco 12 · Testando a função com o exemplo à mão

Como toda função nova, ela é testada com o exemplo de resposta conhecida do bloco 3: duas ações, A e B, que começam a R\\$ 10. No mês seguinte, A vai a R\\$ 15 e B cai para R\\$ 9, e os preços ficam parados no terceiro mês.

1. **Nunca, sem aportes:** R\\$ 200 viram R\\$ 240 e ficam em R\\$ 240.
2. **Mensal, com taxa de 0,1%:** o giro é de R\\$ 30 e o custo, de R\\$ 0,06, então o patrimônio termina em R\\$ 239,94. No terceiro mês, a carteira já está equilibrada e não há nada a vender.
3. **Aporte, com R\\$ 20 por mês:** os dois aportes vão para B, a mais atrasada, que termina com R\\$ 130, enquanto A fica com R\\$ 150.

Repare que o terceiro teste passa `taxa=0.001` pelo nome, e os outros nem mencionam a taxa: vale o padrão, zero.

> **O que esperar:** `[200.0, 240.0, 240.0]`, `30.0 239.94` e `[150.0, 130.0]`.

In [ ]:
precos_teste = pd.DataFrame(
    {"A": [10.0, 15.0, 15.0], "B": [10.0, 9.0, 9.0]},
    index=pd.to_datetime(["2021-10-01", "2021-10-31", "2021-11-30"]),
)

teste1 = simular_carteira(precos_teste, 200, 0, "nunca")
print(teste1["patrimonio"].round(2).tolist())                    # esperado: [200, 240, 240]

teste2 = simular_carteira(precos_teste, 200, 0, "mensal", taxa=0.001)
print(round(teste2["giro"], 2), round(teste2["patrimonio"].iloc[-1], 2))   # esperado: 30.0 239.94

teste3 = simular_carteira(precos_teste, 200, 20, "aporte")
print((teste3["cotas"] * precos_teste.iloc[-1]).round(2).tolist())   # esperado: [150, 130]

### Bloco 13 · As quatro estratégias

Com a função testada, rodamos as quatro estratégias e guardamos cada resultado num dicionário, `resultados`, usando o nome da estratégia como chave.

- `resultados = {}` cria um dicionário vazio.
- `resultados[nome] = ...` cria uma chave nova e guarda o valor nela.
- Um `for` sobre um dicionário percorre as chaves: aqui, os quatro nomes de `estrategias`.

Depois, calculamos o total aportado: o capital inicial mais um aporte em cada uma das 60 datas de fim de mês.

> **O que esperar:** `Total aportado: R$ 14000` e o patrimônio final de cada estratégia. Dentro da f-string, a chave `'patrimonio'` aparece entre aspas simples, porque a f-string inteira já usa aspas duplas.

In [ ]:
resultados = {}
for nome in estrategias:
    resultados[nome] = simular_carteira(precos_decisao, CAPITAL, APORTE_MENSAL, nome)

total_aportado = CAPITAL + APORTE_MENSAL * (len(precos_decisao) - 1)
print("Total aportado: R$", total_aportado)
for nome in resultados:
    print(f"{nome}: R$ {resultados[nome]['patrimonio'].iloc[-1]:.2f}")

**Depois de rodar.** **[Preencher: o patrimônio final de cada estratégia.]**

### Bloco 14 · O patrimônio das quatro estratégias

Para desenhar as quatro curvas, juntamos os patrimônios numa tabela, uma coluna por estratégia, e acrescentamos a rampa do total aportado, como na aula 3. A rampa começa em R\\$ 2.000 e sobe R\\$ 200 por mês.

O título usa o símbolo R\\$ uma vez só: o matplotlib interpreta um texto entre dois cifrões como fórmula matemática, e o título sairia deformado.

A carteira que nunca rebalanceia mantém o azul das aulas anteriores. As outras três ganham cores novas, com estilos de linha diferentes, para continuarem distinguíveis em preto e branco. O laranja e o verde-água ficam reservados para o Ibovespa e o CDI.

> **O que esperar:** cinco linhas subindo juntas. As quatro estratégias ficam muito próximas, e é difícil ver diferenças só pelo gráfico. Por isso, o próximo passo é medir.

In [ ]:
patrimonios = pd.DataFrame()
for nome, r in resultados.items():
    patrimonios[nome] = r["patrimonio"]
patrimonios["total aportado"] = (
    pd.Series(APORTE_MENSAL, index=precos_decisao.index).cumsum() + CAPITAL - APORTE_MENSAL
)

ax = patrimonios.plot(
    figsize=(10, 5),
    style=["-", "--", "-.", ":", "-"],
    color=["#2a78d6", "#e34948", "#6250d6", "#eda100", "#898781"],
    linewidth=2,
    title="Patrimônio das quatro estratégias (R$ 2.000 iniciais + 200 por mês)",
)
ax.set_ylabel("R$")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

**Depois de rodar.** **[Preencher: alguma estratégia se destaca visivelmente?]**

### Bloco 15 · As medidas da aula 2, agora para séries mensais

Para comparar risco e retorno, voltamos às funções da aula 2, aplicadas à **cota** de cada estratégia, e não ao patrimônio, que cresce também com os aportes.

Há um ajuste: a cota é mensal, e não diária. A volatilidade de dados mensais se anualiza com a raiz de 12, e não de 252. Por isso, `volatilidade_anual` e `sharpe` ganham o parâmetro `periodos_por_ano`, com valor padrão de 252. Quem chamar as funções com dados diários, como na aula 2, não precisa mudar nada; para dados mensais, passamos 12.

O teste confere a anualização com uma série que alterna +1% e −1% durante quatro meses. O desvio padrão desses quatro retornos é cerca de 1,15% ao mês, e não 1%, porque a fórmula divide por $n - 1 = 3$, como vimos na aula 2. Multiplicado por $\sqrt{12} \approx 3{,}46$, dá 4,0% ao ano.

> **O que esperar:** `4.0`, a volatilidade anual de quem oscila 1% para cima e para baixo todo mês.

In [ ]:
def retorno_total(serie):
    """Retorno do período inteiro."""
    return serie.iloc[-1] / serie.iloc[0] - 1


def retorno_anual(serie):
    """Retorno médio por ano, com juros compostos (anos contados em dias corridos)."""
    anos = (serie.index[-1] - serie.index[0]).days / 365.25
    return (1 + retorno_total(serie)) ** (1 / anos) - 1


def volatilidade_anual(serie, periodos_por_ano=252):
    """Desvio padrão dos retornos, levado para a escala de um ano."""
    retornos = serie.pct_change().dropna()
    return retornos.std() * periodos_por_ano ** 0.5


def drawdown(serie):
    """Para cada data: quanto a série está abaixo do maior valor já atingido até ali."""
    return serie / serie.cummax() - 1


def queda_maxima(serie):
    """A pior queda do período, do pico ao fundo."""
    return drawdown(serie).min()


def sharpe(serie, cdi, periodos_por_ano=252):
    """Retorno acima do CDI por ano, para cada ponto de volatilidade."""
    excesso = retorno_anual(serie) - retorno_anual(cdi)
    return excesso / volatilidade_anual(serie, periodos_por_ano)


serie_teste = pd.Series([100, 101, 99.99, 100.9899, 99.980001])
print(round(volatilidade_anual(serie_teste, 12) * 100, 1))   # esperado: 4.0

### Bloco 16 · A tabela das estratégias

Agora, a resposta em números. A tabela começa com uma linha por estratégia, `pd.DataFrame(index=list(resultados))`, e é preenchida célula a célula com `.loc[linha, coluna] = valor`. Quando a coluna ainda não existe, o pandas a cria.

- **Patrimônio final** e **ganho sobre o aportado**: o resultado em reais, como na aula 3.
- **Retorno ao ano, volatilidade, queda máxima e Sharpe**: calculados sobre a cota, com `periodos_por_ano=12`.
- **Giro**: quanto foi vendido, somando todos os rebalanceamentos. É zero nas estratégias que não vendem.

> **O que esperar:** uma tabela com quatro linhas e sete colunas. O giro é zero nas estratégias "nunca" e "aporte".

In [ ]:
tabela = pd.DataFrame(index=list(resultados))
for nome, r in resultados.items():
    final = r["patrimonio"].iloc[-1]
    tabela.loc[nome, "Patrimônio final (R$)"] = final
    tabela.loc[nome, "Ganho sobre o aportado (%)"] = (final / total_aportado - 1) * 100
    tabela.loc[nome, "Retorno ao ano (%)"] = retorno_anual(r["cota"]) * 100
    tabela.loc[nome, "Volatilidade ao ano (%)"] = volatilidade_anual(r["cota"], 12) * 100
    tabela.loc[nome, "Queda máxima (%)"] = queda_maxima(r["cota"]) * 100
    tabela.loc[nome, "Sharpe"] = sharpe(r["cota"], cdi_decisao, 12)
    tabela.loc[nome, "Giro (R$)"] = r["giro"]

tabela.round(2)

**Depois de rodar.** **[Preencher com os números: qual estratégia teve o maior patrimônio, a menor volatilidade e o melhor Sharpe?]**

### Bloco 17 · O efeito do custo

Até aqui, negociar não custou nada. Agora, rodamos de novo as duas estratégias que vendem, com a taxa de 0,1% das premissas, e comparamos o patrimônio final com e sem custo.

O dicionário `com_custo` guarda os novos resultados. A tabela mostra o giro, os dois patrimônios e a diferença, que é quanto o custo levou ao longo dos cinco anos. As estratégias "nunca" e "aporte" não aparecem porque não vendem nada: o custo delas seria só o das compras dos aportes, igual nas duas e deixado de fora da comparação.

> **O que esperar:** uma tabela com duas linhas. A estratégia mensal gira muito mais que a anual, e por isso paga mais custo.

In [ ]:
com_custo = {}
for nome in ["mensal", "anual"]:
    com_custo[nome] = simular_carteira(precos_decisao, CAPITAL, APORTE_MENSAL, nome, taxa=TAXA)

custos = pd.DataFrame(index=["mensal", "anual"])
for nome in custos.index:
    custos.loc[nome, "Giro (R$)"] = com_custo[nome]["giro"]
    custos.loc[nome, "Sem custo (R$)"] = resultados[nome]["patrimonio"].iloc[-1]
    custos.loc[nome, "Com custo (R$)"] = com_custo[nome]["patrimonio"].iloc[-1]
custos["Diferença (R$)"] = custos["Sem custo (R$)"] - custos["Com custo (R$)"]

custos.round(2)

**Depois de rodar.** **[Preencher: quanto cada estratégia pagou de custo no período.]**

### Bloco 18 · Quão longe dos 10% cada estratégia terminou?

O objetivo do rebalanceamento é manter os pesos perto do alvo. Calculamos os pesos finais de cada estratégia, a partir das cotas finais que a função devolveu, e medimos o afastamento dos 10% de dois jeitos.

- `pesos_finais[nome] = ...` cria uma coluna por estratégia.
- `(pesos_finais - 10).abs()` é a distância de cada peso até o alvo, em pontos percentuais.
- `.mean()` dá a distância média das 10 ações, e `.max()`, a maior delas.

> **O que esperar:** as duas medidas de afastamento e a tabela de pesos finais. A mensal termina com todos os pesos em 10%, porque acabou de rebalancear. Na distância média, a "aporte" costuma ficar bem mais perto do alvo do que a "nunca", sem ter vendido nada. Na maior distância, nem sempre: o aporte consegue puxar para cima as ações atrasadas, mas não consegue diminuir uma ação que passou muito do alvo. Só a venda faz isso.

In [ ]:
pesos_finais = pd.DataFrame()
for nome, r in resultados.items():
    valores_finais = r["cotas"] * precos_decisao.iloc[-1]
    pesos_finais[nome] = valores_finais / valores_finais.sum() * 100

distancia = (pesos_finais - 10).abs()
print("Distância média até 10% (pontos percentuais):")
print(distancia.mean().round(2))
print()
print("Maior distância até 10% (pontos percentuais):")
print(distancia.max().round(2))
pesos_finais.round(1)

**Depois de rodar.** **[Preencher: a distância média e a maior distância de cada estratégia.]**

## Parte 5 · Validação

### Bloco 19 · A função reproduz as aulas 1 e 3?

A função nova precisa concordar com as contas antigas. São dois testes:

1. **Sem aportes e sem rebalancear**, ela tem de reproduzir a carteira da aula 1 nas 61 datas de decisão.
2. **Sem capital inicial e sem rebalancear**, ela tem de reproduzir a carteira da aula 3, calculada aqui pelo caminho sem laço do bloco 19 daquela aula. A primeira data fica de fora, porque, sem capital inicial, o patrimônio do primeiro dia é zero.

Nos dois casos, a maior diferença tem de ser zero, ou um número minúsculo, por arredondamento.

> **O que esperar:** `Diferença para a aula 1: R$ 0.0` e `Diferença para a aula 3: R$ 0.0`.

In [ ]:
# 1. Sem aportes e sem rebalancear: a carteira da aula 1
so_inicial = simular_carteira(precos_decisao, CAPITAL, 0, "nunca")["patrimonio"]
carteira_aula1 = (precos / precos.iloc[0]).mean(axis=1) * CAPITAL
aula1_decisao = pd.concat([carteira_aula1.iloc[[0]], carteira_aula1.resample("ME").last()])
print("Diferença para a aula 1: R$", round((so_inicial - aula1_decisao).abs().max(), 6))

# 2. Sem capital inicial e sem rebalancear: a carteira da aula 3
so_aportes = simular_carteira(precos_decisao, 0, APORTE_MENSAL, "nunca")["patrimonio"]
cotas_aula3 = (APORTE_MENSAL / len(ACOES) / precos_mes).cumsum()
carteira_aula3 = (cotas_aula3 * precos_mes).sum(axis=1)
print("Diferença para a aula 3: R$", round((so_aportes.iloc[1:] - carteira_aula3).abs().max(), 6))

### Bloco 20 · Depois do rebalanceamento, todos em 10%?

A estratégia mensal rebalanceou no último fim de mês. Então, no fim do período, os pesos têm de somar 100% e valer 10% cada um. Conferimos a soma, o menor e o maior peso.

> **O que esperar:** `Soma dos pesos: 1.0`, `Menor peso: 0.1` e `Maior peso: 0.1`.

In [ ]:
valores_mensal = resultados["mensal"]["cotas"] * precos_decisao.iloc[-1]
pesos_mensal = valores_mensal / valores_mensal.sum()

print("Soma dos pesos:", round(pesos_mensal.sum(), 6))
print("Menor peso:", round(pesos_mensal.min(), 6))
print("Maior peso:", round(pesos_mensal.max(), 6))

## Parte 6 · Interpretação

### Bloco 21 · O que os números dizem

**[Texto a completar com os resultados reais.]**

Com os mesmos R\\$ 14.000 aportados, a estratégia que nunca rebalanceia terminou com **R\\$ [x]**; a mensal, com **R\\$ [x]**; a anual, com **R\\$ [x]**; e a que rebalanceia só com os aportes, com **R\\$ [x]**. A diferença entre a melhor e a pior foi de **R\\$ [x]**, ou **[x]%** do total aportado.

Perguntas que orientam a interpretação:

1. **Rebalancear ajudou ou atrapalhou?** O rebalanceamento vende o que subiu. Se as vencedoras continuaram subindo, ele atrapalhou; se elas devolveram parte da alta, ele ajudou. O resultado depende do período, e não existe garantia para nenhum lado.
2. **E o risco?** Compare a volatilidade e a queda máxima das estratégias. Rebalancear costuma reduzir a concentração, e com ela parte do risco, mesmo quando não aumenta o retorno.
3. **Quanto custou mexer?** Compare o giro e o custo da mensal com os da anual. Rebalancear com muita frequência multiplica as negociações, quase sempre sem melhorar o resultado na mesma proporção.
4. **A estratégia por aporte valeu a pena?** Ela aproximou as ações atrasadas dos 10% sem vender nada, ou seja, sem custo de venda e sem imposto sobre lucro. Seu limite aparece no bloco 18: ela não consegue segurar uma ação que dispara. Para quem aporta todo mês, é um jeito prático de rebalancear: a cada mês, comprar a ação mais atrasada.

## Parte 7 · Limitações

### Bloco 22 · O que esta análise não mostra

- **Um único período.** Cinco anos de um mercado específico. Em outro período, a estratégia vencedora pode ser outra.
- **Dados mensais.** A cota é medida só no fim de cada mês. A queda máxima mensal é menor do que a diária, porque não enxerga os fundos que aconteceram no meio do mês.
- **Custos simplificados.** Uma taxa fixa de 0,1% por operação é uma aproximação: os custos reais variam com a corretora, o volume e a liquidez de cada ação.
- **Imposto fora da conta.** Vendas com lucro podem gerar imposto, o que pesa contra o rebalanceamento por venda. As regras mudam e não foram simuladas.
- **Frações de ação.** Como na aula 3, a simulação compra frações. Com R\\$ 200, a estratégia por aporte compraria, na prática, uma ou poucas ações inteiras por mês.
- **Alvo de pesos iguais.** Todas as estratégias perseguem 10% em cada ação. Outros alvos, como pesos diferentes por setor, mudam o resultado. A aula 6 discute se existe um alvo "melhor".

## Parte 8 · Exercícios

### Bloco 23 · Para praticar

1. **Rebalancear por faixa.** Crie uma estratégia `"faixa"` que só rebalanceia quando algum peso sai da faixa de 5% a 15%. Dica: dentro da função, calcule os pesos antes da decisão e use `rebalancear = (pesos.max() > 0.15) or (pesos.min() < 0.05)`. O `or` ("ou") dá verdadeiro se pelo menos uma das comparações for verdadeira.
2. **Custos maiores.** Mude `TAXA` para 0,005 (0,5%) e rode o bloco 17 de novo. A partir de que custo a estratégia mensal deixa de compensar?
3. **Aporte dividido.** Crie uma versão da estratégia por aporte que reparte os R\\$ 200 entre todas as ações abaixo de 10%, proporcionalmente à distância de cada uma até o alvo.
4. **Outro mês para o anual.** Troque dezembro por setembro (`data.month == 9`), o aniversário da carteira. O resultado muda muito?
5. **Sem aportes.** Rode as quatro estratégias com `APORTE_MENSAL = 0`. O que acontece com a estratégia por aporte? Por quê?

## Resumo

### Bloco 24 · O que aprendemos

**Em finanças:**

- Sem rebalanceamento, os pesos derivam: as ações que sobem passam a pesar mais.
- Rebalancear por venda devolve os pesos ao alvo, mas gera giro, custo e, possivelmente, imposto.
- Rebalancear por aporte aproxima as ações atrasadas do alvo sem vender nada, mas não segura as que disparam.
- Para comparar estratégias com aportes, mede-se o rendimento pela cota, e não pelo patrimônio.
- Rebalancear não garante mais retorno: o efeito depende de as vencedoras continuarem ou não vencendo.

**Em Python:**

- `if`, `elif` e `else` escolhem qual bloco de código roda, a partir de comparações (`>`, `<`, `==`, `!=`...).
- Uma comparação vale verdadeiro ou falso e pode ser guardada numa variável.
- Dicionários guardam pares de chave e valor; `.items()` percorre os pares.
- Parâmetros com valor padrão (`taxa=0`) tornam a função mais flexível sem complicar quem a usa.
- `iloc[[0]]`, com colchetes duplos, devolve uma tabela de uma linha.

## Gancho para a aula 5

### Bloco 25 · Próxima aula

O rebalanceamento só tem o que fazer quando as ações se afastam umas das outras: uma sobe enquanto a outra cai. Se todas subissem e caíssem juntas, na mesma medida, os pesos nunca sairiam dos 10%.

Então, a pergunta seguinte é: as nossas 10 ações andam juntas ou separadas? E isso muda o risco da carteira? Na próxima aula, vamos medir a correlação entre as ações e descobrir quanto risco cai a cada ação acrescentada. E o Python ganha uma ferramenta para testar todas as combinações possíveis de ações.

## Glossário da aula

| Termo | Significado |
|---|---|
| Peso | Fatia de cada ação no patrimônio da carteira |
| Deriva dos pesos | Afastamento natural dos pesos em relação ao alvo, causado pelas variações de preço |
| Rebalanceamento | Trazer os pesos de volta ao alvo, vendendo ou direcionando aportes |
| Giro | Valor vendido num rebalanceamento, igual ao valor comprado |
| Custo de negociar | Taxas e diferenças de preço pagas em cada compra e venda |
| Cota da carteira | Valor que muda só com o rendimento, sem o efeito dos aportes |
| `if` / `elif` / `else` | Estrutura que escolhe qual bloco de código rodar, a partir de condições |
| Comparação | Expressão que vale verdadeiro ou falso, como `peso > 0.10` |
| Dicionário | Estrutura que guarda pares de chave e valor, escrita entre chaves |
| Valor padrão | Valor que um parâmetro assume quando não é informado |

---

*Python Finance Lab · Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*